# Notebook 1 - Data Cleaning of the CICIoMT2024 Subset

Run this notebook **first**. It produces the cleaned files that Notebook 2 (experiments) uses.

### Cleaning protocol

We follow a documented, rule-based cleaning protocol. Each step has a fixed rule, and we record how many rows or columns each step removes, so the process can be reported and repeated.

| Step | Check | Rule | Applied to |
|---|---|---|---|
| C1 | Schema | same columns, all numeric, label from file name | train & test |
| C2 | Missing values | remove rows with any missing value | train & test |
| C3 | Infinite values | remove rows with ±∞ | train & test |
| C4 | Domain validity | remove physically impossible rows (negative times/counts/sizes, 0–1 features outside [0, 1], Min > Max) | train & test |
| C5 | Exact duplicate rows | keep the first copy | each set separately |
| C6 | Label conflicts | identical features but different labels → remove all copies | each set separately |
| C7 | Train–test overlap | test rows identical to a training row → remove from test | test |
| C8 | Zero-variance features | feature with one value in **training** data → drop | decided on train, applied to both |
| C9 | Duplicate features | feature identical to another in **training** data → drop the copy | decided on train, applied to both |
| C10 | Outliers | detect and **report** (IQR rule), **not removed** | train & test |

**Why train and test are kept separate:** the dataset authors provide them as separate captures. Merging and re-splitting at random could place almost identical records from the same capture session on both sides, which inflates the scores (data leakage). Every rule that "learns" something (C8, C9) is learned from the **training data only**.

**Why outliers are not removed:** in intrusion detection, extreme values are often the attack itself (for example, a flood creates extreme packet rates). Deleting them would delete attack evidence. The Random Forest used later does not need the data to be normally distributed.

## Step 0 — Tools and folders

In [1]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "."                     # where to look for the CSV files ("." = this notebook's folder and all sub-folders)
OUT      = "cleaned_data"          # cleaned files and the cleaning report go here
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "font.family": "serif", "axes.grid": True, "grid.alpha": 0.3})

log = []        # every step writes one line here: the cleaning log
def record(step, description, train_before, train_after, test_before, test_after):
    log.append({"Step": step, "Description": description,
                "Train before": train_before, "Train removed": train_before - train_after, "Train after": train_after,
                "Test before": test_before, "Test removed": test_before - test_after, "Test after": test_after})
    print(f"{step}: {description:55s} train -{train_before - train_after:,}  test -{test_before - test_after:,}")

## C1 — Load the files and check the schema

The label is taken from the file name, for example `ARP_Spoofing_train.pcap.csv` → `ARP_Spoofing`. Split files such as `TCP_IP-DDoS-SYN1 … SYN4` are joined into one class.

In [2]:
def label_from_filename(path):
    name = re.sub(r"_(train|test)\.pcap\.csv$", "", os.path.basename(path), flags=re.IGNORECASE)
    return re.sub(r"\d+$", "", name)

def find_files(kind):
    # Finds every *_train.pcap.csv or *_test.pcap.csv file, in any folder below DATA_DIR.
    # So it does not matter whether your folders are called Train/Test, train/test or something else.
    files = [f for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
             if f.lower().endswith(f"_{kind}.pcap.csv") and "cleaned_data" not in f]
    return sorted(files)

def load_folder(kind):
    files = find_files(kind)
    if not files:
        raise FileNotFoundError(
            f"No '*_{kind}.pcap.csv' files found under: {os.path.abspath(DATA_DIR)}\n"
            f"Files/folders here: {os.listdir(DATA_DIR)}\n"
            "Put this notebook in the same folder as your Train and Test folders, "
            "or set DATA_DIR in Step 0 to the folder that contains them.")
    print(f"{kind.upper()} files found: {len(files)}")
    parts, columns = [], None
    for f in files:
        print("   ", f)
        d = pd.read_csv(f)
        if columns is None: columns = list(d.columns)
        assert list(d.columns) == columns, f"Column mismatch in {f}"
        d = d.apply(pd.to_numeric, errors="coerce")    # anything non-numeric becomes missing (caught in C2)
        d["Label"] = label_from_filename(f)
        parts.append(d)
    return pd.concat(parts, ignore_index=True), columns

train, cols_tr = load_folder("train")
test,  cols_te = load_folder("test")
assert cols_tr == cols_te, "Train and test have different columns"
FEATURES = cols_tr

common = sorted(set(train["Label"]) & set(test["Label"]))
train = train[train["Label"].isin(common)].reset_index(drop=True)
test  = test[test["Label"].isin(common)].reset_index(drop=True)

raw_counts = pd.DataFrame({"Train (raw)": train["Label"].value_counts(),
                           "Test (raw)": test["Label"].value_counts()}).reindex(common)
record("C1", "Load files, check schema (rows loaded)", len(train), len(train), len(test), len(test))
print("\nFeatures:", len(FEATURES), "| Classes:", len(common))
raw_counts

TRAIN files found: 14
    .\Train\ARP_Spoofing_train.pcap.csv
    .\Train\Benign_train.pcap.csv
    .\Train\MQTT-DDoS-Connect_Flood_train.pcap.csv
    .\Train\MQTT-Malformed_Data_train.pcap.csv
    .\Train\Recon-Port_Scan_train.pcap.csv
    .\Train\Recon-VulScan_train.pcap.csv
    .\Train\TCP_IP-DDoS-SYN1_train.pcap.csv
    .\Train\TCP_IP-DDoS-SYN2_train.pcap.csv
    .\Train\TCP_IP-DDoS-SYN3_train.pcap.csv
    .\Train\TCP_IP-DDoS-SYN4_train.pcap.csv
    .\Train\TCP_IP-DoS-SYN1_train.pcap.csv
    .\Train\TCP_IP-DoS-SYN2_train.pcap.csv
    .\Train\TCP_IP-DoS-SYN3_train.pcap.csv
    .\Train\TCP_IP-DoS-SYN4_train.pcap.csv
TEST files found: 8
    .\Test\ARP_Spoofing_test.pcap.csv
    .\Test\Benign_test.pcap.csv
    .\Test\MQTT-DDoS-Connect_Flood_test.pcap.csv
    .\Test\MQTT-Malformed_Data_test.pcap.csv
    .\Test\Recon-Port_Scan_test.pcap.csv
    .\Test\Recon-VulScan_test.pcap.csv
    .\Test\TCP_IP-DDoS-SYN_test.pcap.csv
    .\Test\TCP_IP-DoS-SYN_test.pcap.csv
C1: Load files, check schema 

,Train (raw),Test (raw)
Label,,
ARP_Spoofing,16047,1744
Benign,192732,37607
MQTT-DDoS-Connect_Flood,173036,41916
MQTT-Malformed_Data,5130,1747
Recon-Port_Scan,83981,22622
Recon-VulScan,2173,1034
TCP_IP-DDoS-SYN,801962,172397
TCP_IP-DoS-SYN,441903,98595


## C2 — Missing values

In [3]:
a, b = len(train), len(test)
print("Missing cells — train:", int(train[FEATURES].isna().sum().sum()), " test:", int(test[FEATURES].isna().sum().sum()))
train = train.dropna().reset_index(drop=True)
test  = test.dropna().reset_index(drop=True)
record("C2", "Remove rows with missing values", a, len(train), b, len(test))

Missing cells — train: 0  test: 0
C2: Remove rows with missing values                         train -0  test -0


## C3 — Infinite values

In [4]:
a, b = len(train), len(test)
finite_tr = np.isfinite(train[FEATURES].values).all(axis=1)
finite_te = np.isfinite(test[FEATURES].values).all(axis=1)
train, test = train[finite_tr].reset_index(drop=True), test[finite_te].reset_index(drop=True)
record("C3", "Remove rows with infinite values", a, len(train), b, len(test))

C3: Remove rows with infinite values                        train -0  test -0


## C4 — Domain validity (physically impossible values)

| Rule | Why |
|---|---|
| Times, counts, sizes, rates must be **≥ 0** | a negative duration or packet size cannot happen |
| Flag and protocol indicators must lie in **[0, 1]** | they are averaged 0/1 indicators |
| **Min ≤ AVG ≤ Max** (packet sizes) | the average cannot fall outside its own minimum and maximum |

In [5]:
UNIT_INTERVAL = [c for c in FEATURES if c.endswith("_flag_number")] + \
    ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"]
UNIT_INTERVAL = [c for c in UNIT_INTERVAL if c in FEATURES]
NON_NEGATIVE = FEATURES     # every feature in this dataset is a time, count, size, rate or indicator

def invalid_mask(d):
    neg   = (d[NON_NEGATIVE] < 0).any(axis=1)
    unit  = ((d[UNIT_INTERVAL] < 0) | (d[UNIT_INTERVAL] > 1)).any(axis=1)
    order = (d["Min"] > d["AVG"] + 1e-9) | (d["AVG"] > d["Max"] + 1e-9)
    return neg | unit | order, neg, unit, order

for name, d in [("Train", train), ("Test", test)]:
    m, neg, unit, order = invalid_mask(d)
    print(f"{name}: negative values={int(neg.sum())}, 0-1 features outside [0,1]={int(unit.sum())}, Min/AVG/Max order broken={int(order.sum())}")
    bad_cols = [c for c in NON_NEGATIVE if (d[c] < 0).any()]
    if bad_cols: print("   columns with negative values:", bad_cols)

a, b = len(train), len(test)
train = train[~invalid_mask(train)[0]].reset_index(drop=True)
test  = test[~invalid_mask(test)[0]].reset_index(drop=True)
record("C4", "Remove physically impossible rows", a, len(train), b, len(test))

Train: negative values=1, 0-1 features outside [0,1]=0, Min/AVG/Max order broken=0
   columns with negative values: ['IAT']
Test: negative values=0, 0-1 features outside [0,1]=0, Min/AVG/Max order broken=0
C4: Remove physically impossible rows                       train -1  test -0


## C5 — Exact duplicate rows

Network flows often repeat exactly (especially floods). Duplicates make the model see the same example many times and can make test scores look better than they are. We keep one copy of each row.

In [6]:
a, b = len(train), len(test)
dup_by_class = pd.DataFrame({"Train duplicates": train[train.duplicated()]["Label"].value_counts(),
                             "Test duplicates":  test[test.duplicated()]["Label"].value_counts()}).reindex(common).fillna(0).astype(int)
train = train.drop_duplicates().reset_index(drop=True)
test  = test.drop_duplicates().reset_index(drop=True)
record("C5", "Remove exact duplicate rows", a, len(train), b, len(test))
dup_by_class

C5: Remove exact duplicate rows                             train -3,794  test -1,724


,Train duplicates,Test duplicates
Label,,
ARP_Spoofing,0,0
Benign,0,0
MQTT-DDoS-Connect_Flood,0,0
MQTT-Malformed_Data,0,0
Recon-Port_Scan,3734,1701
Recon-VulScan,44,23
TCP_IP-DDoS-SYN,16,0
TCP_IP-DoS-SYN,0,0


## C6 — Label conflicts

If **exactly the same feature values** appear with **two different labels**, no model can classify them correctly, and keeping them adds label noise. We remove all copies of such rows.

In [7]:
def remove_conflicts(d):
    h = pd.util.hash_pandas_object(d[FEATURES], index=False)
    n_labels = d.groupby(h.values)["Label"].transform("nunique")
    conflict = n_labels.values > 1
    return d[~conflict].reset_index(drop=True), d[conflict]["Label"].value_counts()

a, b = len(train), len(test)
train, conf_tr = remove_conflicts(train)
test,  conf_te = remove_conflicts(test)
record("C6", "Remove rows with conflicting labels", a, len(train), b, len(test))
pd.DataFrame({"Train conflicts": conf_tr, "Test conflicts": conf_te}).reindex(common).fillna(0).astype(int)

C6: Remove rows with conflicting labels                     train -0  test -0


,Train conflicts,Test conflicts
Label,,
ARP_Spoofing,0,0
Benign,0,0
MQTT-DDoS-Connect_Flood,0,0
MQTT-Malformed_Data,0,0
Recon-Port_Scan,0,0
Recon-VulScan,0,0
TCP_IP-DDoS-SYN,0,0
TCP_IP-DoS-SYN,0,0


## C7 — Train–test overlap

A test row that is identical to a training row is not an "unseen" example. We remove it from the **test** set only.

In [8]:
a, b = len(train), len(test)
h_train = set(pd.util.hash_pandas_object(train[FEATURES], index=False))
in_train = pd.util.hash_pandas_object(test[FEATURES], index=False).isin(h_train).values
print("Overlap by class:"); print(test[in_train]["Label"].value_counts())
test = test[~in_train].reset_index(drop=True)
record("C7", "Remove test rows also present in train", a, len(train), b, len(test))

Overlap by class:
Label
Recon-Port_Scan    195
Name: count, dtype: int64
C7: Remove test rows also present in train                  train -0  test -195


## C8 — Zero-variance features

A feature that has only **one value in the whole training set** cannot help separate classes. The decision is made on the **training data** and applied to both sets.

In [9]:
zero_var = [c for c in FEATURES if train[c].nunique() <= 1]
print("Zero-variance features (dropped):", zero_var if zero_var else "none")
FEATURES = [c for c in FEATURES if c not in zero_var]

Zero-variance features (dropped): ['Drate']


## C9 — Duplicate features

Two columns that are **identical in every training row** carry the same information. We keep the first and drop the copy.

In [10]:
dup_features = []
for i, a_col in enumerate(FEATURES):
    for b_col in FEATURES[i + 1:]:
        if b_col not in [d for _, d in dup_features] and train[a_col].equals(train[b_col]):
            dup_features.append((a_col, b_col))
for keep, drop in dup_features:
    agree_test = (test[keep] == test[drop]).mean() * 100
    print(f"'{drop}' is identical to '{keep}' in train ({agree_test:.1f}% identical in test) -> drop '{drop}'")
FEATURES = [c for c in FEATURES if c not in [d for _, d in dup_features]]
print("Features remaining:", len(FEATURES))

'Srate' is identical to 'Rate' in train (100.0% identical in test) -> drop 'Srate'
'LLC' is identical to 'IPv' in train (100.0% identical in test) -> drop 'LLC'
Features remaining: 42


## C10 — Outliers (detected and reported, not removed)

We use the standard **IQR rule**: a value is an outlier if it lies below Q1 − 1.5·IQR or above Q3 + 1.5·IQR. The limits are computed on the training data: once over all classes together (per-feature table) and once **within each class** (per-class table), because classes differ so much that pooled limits flag almost every row of the small classes.

Features with **IQR = 0** (mostly-zero indicators and counts) are left out of this check, because there every non-zero value would count as an "outlier", which is meaningless.

In [11]:
q1, q3 = train[FEATURES].quantile(0.25), train[FEATURES].quantile(0.75)
iqr = q3 - q1
CHECKED = [c for c in FEATURES if iqr[c] > 0]
print(f"Outlier check applied to {len(CHECKED)} features with IQR > 0;",
      f"{len(FEATURES) - len(CHECKED)} features with IQR = 0 skipped.")
low, high = q1[CHECKED] - 1.5 * iqr[CHECKED], q3[CHECKED] + 1.5 * iqr[CHECKED]
is_out = (train[CHECKED] < low) | (train[CHECKED] > high)

outlier_by_feature = (is_out.mean() * 100).sort_values(ascending=False).rename("% outlying values (train)")
# Within each class: IQR limits computed from that class's own training rows
within = {}
for lbl, g in train.groupby("Label"):
    q1g, q3g = g[FEATURES].quantile(0.25), g[FEATURES].quantile(0.75)
    iqg = q3g - q1g
    chk = [c for c in FEATURES if iqg[c] > 0]
    if not chk: within[lbl] = 0.0; continue
    o = (g[chk] < q1g[chk] - 1.5 * iqg[chk]) | (g[chk] > q3g[chk] + 1.5 * iqg[chk])
    within[lbl] = o.values.mean() * 100
outlier_by_class = pd.Series(within).reindex(common).rename("% outlying values within class (train)")
print("Outliers are kept. Overall share of outlying values (pooled limits):", f"{is_out.values.mean() * 100:.1f}%")
display(outlier_by_class.round(1).to_frame())
outlier_by_feature.head(15).round(2).to_frame()

Outlier check applied to 19 features with IQR > 0; 23 features with IQR = 0 skipped.
Outliers are kept. Overall share of outlying values (pooled limits): 12.0%


,% outlying values within class (train)
ARP_Spoofing,2.8
Benign,11.9
MQTT-DDoS-Connect_Flood,2.3
MQTT-Malformed_Data,2.0
Recon-Port_Scan,9.0
Recon-VulScan,5.6
TCP_IP-DDoS-SYN,3.3
TCP_IP-DoS-SYN,8.2


,% outlying values (train)
Tot sum,27.72
rst_count,22.47
Covariance,18.31
IAT,17.42
Header_Length,16.90
Min,16.76
Magnitue,16.03
Tot size,15.70
AVG,15.42
Radius,14.80


## Summary — the cleaning log and class counts

In [12]:
cleaning_log = pd.DataFrame(log)
clean_counts = pd.DataFrame({"Train (clean)": train["Label"].value_counts(),
                             "Test (clean)": test["Label"].value_counts()}).reindex(common)
class_summary = raw_counts.join(clean_counts)
class_summary["Train kept %"] = (100 * class_summary["Train (clean)"] / class_summary["Train (raw)"]).round(1)
class_summary["Test kept %"]  = (100 * class_summary["Test (clean)"]  / class_summary["Test (raw)"]).round(1)
display(cleaning_log)
class_summary

,Step,Description,Train before,Train removed,Train after,Test before,Test removed,Test after
0,C1,"Load files, check schema (rows loaded)",1716964,0,1716964,377662,0,377662
1,C2,Remove rows with missing values,1716964,0,1716964,377662,0,377662
2,C3,Remove rows with infinite values,1716964,0,1716964,377662,0,377662
3,C4,Remove physically impossible rows,1716964,1,1716963,377662,0,377662
4,C5,Remove exact duplicate rows,1716963,3794,1713169,377662,1724,375938
5,C6,Remove rows with conflicting labels,1713169,0,1713169,375938,0,375938
6,C7,Remove test rows also present in train,1713169,0,1713169,375938,195,375743


,Train (raw),Test (raw),Train (clean),Test (clean),Train kept %,Test kept %
Label,,,,,,
ARP_Spoofing,16047,1744,16047,1744,100.0,100.0
Benign,192732,37607,192731,37607,100.0,100.0
MQTT-DDoS-Connect_Flood,173036,41916,173036,41916,100.0,100.0
MQTT-Malformed_Data,5130,1747,5130,1747,100.0,100.0
Recon-Port_Scan,83981,22622,80247,20726,95.6,91.6
Recon-VulScan,2173,1034,2129,1011,98.0,97.8
TCP_IP-DDoS-SYN,801962,172397,801946,172397,100.0,100.0
TCP_IP-DoS-SYN,441903,98595,441903,98595,100.0,100.0


In [ ]:
# Figure 1: rows remaining after each step
fig, ax = plt.subplots(figsize=(7, 3.8))
steps = cleaning_log["Step"]
ax.plot(steps, cleaning_log["Train after"], "o-", label="Train", color="#1F4E78")
ax.plot(steps, cleaning_log["Test after"],  "s-", label="Test",  color="#ED7D31")
ax.set_yscale("log"); ax.set_ylabel("Rows remaining (log scale)"); ax.legend()
ax.set_title("Figure 1. Rows remaining after each cleaning step")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_cleaning_flow.png"); plt.show()

# Figure 2: class sizes before and after cleaning
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, s in zip(axes, ["Train", "Test"]):
    cs = class_summary[[f"{s} (raw)", f"{s} (clean)"]]
    y = np.arange(len(cs))
    ax.barh(y + 0.2, cs.iloc[:, 0], 0.4, label="Raw", color="#BFBFBF")
    ax.barh(y - 0.2, cs.iloc[:, 1], 0.4, label="Clean", color="#1F4E78")
    ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels(cs.index, fontsize=8)
    ax.set_title(s); ax.set_xlabel("Rows (log scale)"); ax.legend(fontsize=8)
fig.suptitle("Figure 2. Class sizes before and after cleaning")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_class_sizes.png"); plt.show()

## Save the cleaned data and the cleaning report

- `cleaned_data/train_clean.csv`, `cleaned_data/test_clean.csv` – used by Notebook 2
- `cleaned_data/cleaning_report.xlsx` – all tables
- `cleaned_data/cleaning_summary.txt` – **send this back** for the Methods section

In [ ]:
train[FEATURES + ["Label"]].to_csv(f"{OUT}/train_clean.csv", index=False)
test[FEATURES + ["Label"]].to_csv(f"{OUT}/test_clean.csv", index=False)

with pd.ExcelWriter(f"{OUT}/cleaning_report.xlsx") as xw:
    cleaning_log.to_excel(xw, sheet_name="Cleaning_log", index=False)
    class_summary.to_excel(xw, sheet_name="Class_counts")
    dup_by_class.to_excel(xw, sheet_name="Duplicates_by_class")
    pd.DataFrame(dup_features, columns=["kept", "dropped"]).to_excel(xw, sheet_name="Duplicate_features", index=False)
    pd.Series(zero_var, dtype=str).to_frame("zero_variance_dropped").to_excel(xw, sheet_name="Zero_variance", index=False)
    outlier_by_feature.to_frame().to_excel(xw, sheet_name="Outliers_by_feature")
    outlier_by_class.to_frame().to_excel(xw, sheet_name="Outliers_by_class")

pd.set_option("display.width", 250)
text = "\n".join([
    "DATA CLEANING SUMMARY — CICIoMT2024 subset", "=" * 60,
    "Train and test kept separate (official split). Rules learned on train only.",
    "\nCleaning log:\n" + cleaning_log.to_string(index=False),
    "\nZero-variance features dropped: " + (", ".join(zero_var) if zero_var else "none"),
    "Duplicate features dropped: " + (", ".join(f"{d} (= {k})" for k, d in dup_features) if dup_features else "none"),
    f"Final number of features: {len(FEATURES)}",
    "Final features: " + ", ".join(FEATURES),
    "\nClass counts:\n" + class_summary.to_string(),
    "\nOutliers (kept): % outlying values within each class (class-specific IQR limits):\n" + outlier_by_class.round(1).to_string(),
])
with open(f"{OUT}/cleaning_summary.txt", "w") as fh: fh.write(text)
print(text)